# Word-level alignment (WhisperX)

Needs a GPU: **Runtime > Change runtime type > GPU**.

Feeds the KNOWN poem text (`bateau_ivre.yaml`) into the alignment rather than trusting Whisper's own transcription: Rimbaud's archaic/poetic French, sung or rapped, is exactly the kind of input ASR mishears -- forced alignment of the real words against the audio sidesteps that entirely. Whisper's own `transcribe()` pass is used only to find rough segment boundaries (it's good at that -- real voice-activity timing, not a guess), then each segment's recognized text is swapped for the matching known line before the alignment model runs on it.

In [ ]:
!pip install -q whisperx

In [ ]:
from google.colab import files
uploaded = files.upload()  # select the audio file
AUDIO_PATH = next(iter(uploaded))

In [ ]:
uploaded2 = files.upload()  # select bateau_ivre.yaml
YAML_PATH = next(iter(uploaded2))

In [ ]:
import yaml


def load_known_lines(yaml_path):
    sections = yaml.safe_load(open(yaml_path, encoding="utf-8"))
    by_number = {}
    for entry in sections:
        for strophe in entry["strophes"]:
            by_number[strophe["strophe"]] = strophe
    lines = []
    for n in sorted(by_number):
        lines.extend(by_number[n]["vers"])
    return lines


known_lines = load_known_lines(YAML_PATH)
print(f"{len(known_lines)} known lines loaded")

In [ ]:
import whisperx

device = "cuda"
batch_size = 16
compute_type = "float16"  # try "int8" if you hit GPU memory limits

model = whisperx.load_model("large-v2", device, compute_type=compute_type, language="fr")
audio = whisperx.load_audio(AUDIO_PATH)
result = model.transcribe(audio, batch_size=batch_size, language="fr")

print(f"Whisper found {len(result['segments'])} segments vs {len(known_lines)} known lines")
for seg in result["segments"][:8]:
    print(f"  {seg['start']:6.2f}-{seg['end']:6.2f}  {seg['text']}")

**Whisper's segments won't land one-per-poem-line** (it splits on pauses in the performance, not on verse boundaries) -- so a blind `zip()` between segments and known lines would misassign text from the first mismatch onward, with no error, just silently wrong timestamps for everything after. The next cell instead counts *words* per segment (Whisper's transcribed words are often wrong individually, but the word *count* tracks the real audio well) and consumes that many known lines per segment -- snapped to whole line boundaries, never mid-line -- so each segment gets a correctly worded, correctly sized block of the real poem text instead of Whisper's guess. **Check the printed per-segment text below against the poem before running the alignment cell after it.**

In [ ]:
import re


def word_count(s):
    return len(re.findall(r"\w+", s, flags=re.UNICODE))


known_word_counts = [word_count(l) for l in known_lines]
line_idx = 0
for seg in result["segments"]:
    target = word_count(seg["text"])
    start_idx = line_idx
    acc = 0
    while line_idx < len(known_lines):
        next_acc = acc + known_word_counts[line_idx]
        if acc > 0 and abs(next_acc - target) > abs(acc - target):
            break
        acc = next_acc
        line_idx += 1
    seg["text"] = " ".join(known_lines[start_idx:line_idx])

# Leftover known lines (rounding at the very end) go on the last segment.
if line_idx < len(known_lines):
    result["segments"][-1]["text"] += " " + " ".join(known_lines[line_idx:])

for seg in result["segments"]:
    print(f"{seg['start']:7.2f}-{seg['end']:7.2f}  {seg['text']}")

align_model, metadata = whisperx.load_align_model(language_code="fr", device=device)
result = whisperx.align(result["segments"], align_model, metadata, audio, device, return_char_alignments=False)

In [ ]:
import json

words = [
    {"word": w["word"].strip(), "start": w.get("start"), "end": w.get("end")}
    for seg in result["segments"]
    for w in seg["words"]
]
with open("word_timestamps.json", "w", encoding="utf-8") as f:
    json.dump(words, f, ensure_ascii=False, indent=2)

files.download("word_timestamps.json")